In [1]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

In [2]:
mnist = fetch_openml("mnist_784", version=1)

In [3]:
type(mnist.data)

pandas.core.frame.DataFrame

In [4]:
type(mnist.target)

pandas.core.series.Series

In [5]:
mnist.data.shape

(70000, 784)

In [6]:
mnist.data.iloc[0].to_numpy()

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   3,  18,  18,  18,
       126, 136, 175,  26, 166, 255, 247, 127,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,  30,  36,  94, 154, 17

In [7]:
from skimage.feature import hog
def extract_feature_with_hog(X):
    tranformeds = []
    for i, img in enumerate(X):
        tranformed = hog(img.reshape(28, 28),
                         orientations=9,
                         pixels_per_cell=(7, 7),
                         cells_per_block=(3, 3))
        tranformeds.append(tranformed)
    return np.asarray(tranformeds)


In [8]:
X = extract_feature_with_hog(mnist.data.to_numpy())
y = mnist.target

In [9]:
X.shape

(70000, 324)

In [11]:
X[:3]

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.19932195e-02,
        2.57664429e-01, 2.08770880e-01, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        9.52931538e-02, 1.82559448e-01, 2.57664429e-01, 7.69865418e-02,
        2.28247473e-02, 0.00000000e+00, 0.00000000e+00, 1.39858159e-02,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        1.28403152e-01, 6.09062694e-02, 7.32888728e-02, 0.00000000e+00,
        2.49710930e-01, 7.78217301e-02, 1.94535806e-01, 2.57664429e-01,
        1.00636564e-01, 8.49373746e-02, 1.09315948e-01, 1.18233822e-01,
        1.05067072e-01, 2.57275481e-01, 2.33276975e-01, 2.27080019e-01,
        4.22673428e-02, 0.00000000e+00, 0.00000000e+00, 0.000000

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
model = RandomForestClassifier(n_estimators=100, n_jobs=-1)

In [14]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
predicts = model.predict(X_test)
accuracy = accuracy_score(predicts, y_test)
print(f"Akurasi : {accuracy}")

Akurasi : 0.9640714285714286


In [17]:
import cv2
import numpy as np
from skimage.feature import hog

img = cv2.imread("../seven.png")
print(type(img))
print(img.shape)

def center_image(img):
    """Center digit berdasarkan bounding box piksel aktif"""
    # Threshold untuk temukan piksel aktif
    _, thresh = cv2.threshold(img, 25, 255, cv2.THRESH_BINARY)
    
    # Temukan bounding box dari piksel aktif
    coords = cv2.findNonZero(thresh)
    x, y, w, h = cv2.boundingRect(coords)
    
    # Crop digit
    cropped = img[y:y+h, x:x+w]
    cv2.imwrite("centered_nopad.png", cropped)
    
    # Resize crop agar muat di canvas 20x20 (sisakan 4px padding)
    scale = min(20 / h, 20 / w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # Tempatkan di tengah canvas 28x28
    canvas = np.zeros((28, 28), dtype=np.uint8)
    row_start = (28 - new_h) // 2
    col_start = (28 - new_w) // 2
    canvas[row_start:row_start+new_h, col_start:col_start+new_w] = resized
    
    cv2.imwrite("centered.png", canvas)

    return canvas

def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    
    # Konversi ke grayscale
    if img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Invert jika background putih (sebelum centering)
    if img.mean() > 127:
        img = 255 - img

    # ✅ Center digit
    img = center_image(img)

    img_flat = img.flatten().reshape(1, -1)
    img_flat = extract_feature_with_hog(img_flat)
    return model.predict(img_flat)

<class 'numpy.ndarray'>
(600, 1000, 3)


In [19]:
predict_image("../three.png")

array(['3'], dtype=object)